<a href="https://colab.research.google.com/github/SattamAltwaim/StarX/blob/main/experiments/10_build_sketch_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup: clone the StarX repo and the pinned TripoSR commit, install this
# notebook's dependencies, mount Drive, and report what machine we are on.
import os
import subprocess
import sys

BRANCH = "main"
TRIPOSR_COMMIT = "107cefdc244c39106fa830359024f6a2f1c78871"
NOTEBOOK_ID = "10"

IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    REPO_DIR = "/content/StarX"
    if not os.path.exists(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--branch", BRANCH,
             "https://github.com/SattamAltwaim/StarX.git", REPO_DIR],
            check=True,
        )
    TRIPOSR_DIR = "/content/TripoSR"
else:
    # off Colab the kernel's cwd is unpredictable (VS Code often starts in
    # $HOME) - walk up from the notebook location and the cwd to find the
    # repo, with ~/StarX as the final fallback
    def _find_repo():
        candidates = [globals().get("__vsc_ipynb_file__"), os.getcwd()]
        for start in candidates:
            if not start:
                continue
            path = os.path.abspath(
                os.path.dirname(start) if os.path.isfile(start) else start
            )
            while path != os.path.dirname(path):
                if os.path.exists(os.path.join(path, "starx", "pins.py")):
                    return path
                path = os.path.dirname(path)
        home_repo = os.path.join(os.path.expanduser("~"), "StarX")
        if os.path.exists(os.path.join(home_repo, "starx", "pins.py")):
            return home_repo
        raise RuntimeError(
            "could not locate the StarX repo - start Jupyter inside it "
            "or clone it to ~/StarX"
        )

    REPO_DIR = _find_repo()
    TRIPOSR_DIR = os.path.join(REPO_DIR, "third_party", "TripoSR")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from starx import pins

assert pins.TRIPOSR_COMMIT == TRIPOSR_COMMIT, "notebook pin out of sync with starx/pins.py"
if pins.PIP_PINS[NOTEBOOK_ID]:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *pins.PIP_PINS[NOTEBOOK_ID]],
        check=True,
    )

from starx import colab as scolab

DRIVE = scolab.mount_drive()
report = scolab.setup_report()

# 10 - Building the synthetic sketch dataset

Notebook 09 edge-detects a render inside the training step. That is fine for one run, but it recomputes the same drawings every epoch and leaves nothing to inspect. This notebook writes them down once, for every design and every stored view, so that training becomes an ordinary supervised job: a dataset on disk, a torch `DataLoader` with workers, and a batch dimension.

A sample is a **(design, input view) pair**, not a design. Each design was rendered from sixteen camera positions in notebook 03, and every one of those becomes an input in its own right - so the designs on Drive turn into roughly a hundred and forty thousand training samples, each a drawing of a real part from a known viewpoint.

The dataset rides alongside the design shards rather than replacing them. The renders, masks and cameras stay where they are; a parallel set of shards adds one grayscale drawing per design and view, and both extract into the same local cache. Nothing is duplicated and notebook 03 never has to run again.

The steps: check what is on Drive, look at one design's drawings before committing, build both splits, then reopen the result as the `Dataset` class the training script will use and look at what comes out of it.

Runs anywhere the shards are reachable - Colab with Drive mounted, the Mac against its synced Drive folder, or Ibex. It needs no GPU, though it will use one to go faster.

In [ ]:
# Configuration - every tunable for this notebook lives here.
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from starx import shards, sketchdata, synth
from starx.config import StarXConfig, shard_dir, sketch_shard_dir

SMOKE = False   # True: build only the 20-design smoke shards, as a rehearsal
SEED = 1337
SPLITS = ["smoke_train", "smoke_test"] if SMOKE else ["train", "test"]

cfg = StarXConfig(
    drive_root=str(DRIVE / "StarX")
    if DRIVE is not None
    else os.path.join(REPO_DIR, "data", "StarX"),
    local_root="/content/starx_local"
    if IN_COLAB
    else os.path.join(REPO_DIR, "data", "local"),
    # the edge detector - these four values ARE the dataset, and are
    # written next to the shards so a later config edit cannot silently
    # train on drawings made with different settings
    sketch_size=512,        # TripoSR's native input resolution
    edge_blur_sigma=1.2,
    edge_gain=3.0,
    edge_bg=1.0,            # 1.0 = white page, 0.5 = TripoSR's composite gray
    # how many ground-truth views a sample carries for supervision
    supervision_views=4,
)
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"data root: {cfg.drive_root}")
print(f"splits:    {SPLITS}")
print(f"edge:      {sketchdata.edge_params(cfg)}")
print(f"device:    {device}")

In [ ]:
# What is already on Drive, before anything is written.
for split in SPLITS:
    source = shard_dir(cfg, split)
    done = shards.list_done_shards(source)
    designs = sum(
        json.loads(shards.marker_path(p).read_text())["n_samples"] for p in done
    )
    existing = shards.list_done_shards(
        sketch_shard_dir(cfg, split), sketchdata.SKETCH_PREFIX
    )
    print(f"{split:12s} {len(done):3d} design shards, {designs:5d} designs"
          f"   sketch shards already built: {len(existing)}")
    assert done, f"no design shards under {source} - run notebook 03 first"

In [ ]:
# Look before building: one design straight out of a design shard, its
# stored renders above the drawings they will become. Reading a single tar
# keeps this cheap even when the shards live on Drive.
preview_tar = shards.list_done_shards(shard_dir(cfg, SPLITS[0]))[0]
preview = next(shards.iter_shard(preview_tar))
preview_sketches = synth.sketches_for_views(preview["views"], cfg, device).cpu()

n_show = 8
fig, axes = plt.subplots(2, n_show, figsize=(2.0 * n_show, 4.4))
for c in range(n_show):
    axes[0, c].imshow(preview["views"][c])
    axes[0, c].set_title(
        f"az {preview['meta']['view_angles'][c][0]:.0f}", fontsize=8
    )
    axes[1, c].imshow(preview_sketches[c], cmap="gray", vmin=0, vmax=1)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
axes[0, 0].set_ylabel("render", fontsize=9)
axes[1, 0].set_ylabel("sketch", fontsize=9)
fig.tight_layout()
plt.show()

print(f"{preview['design_id']}: {preview['meta']['n_views']} views -> "
      f"{preview['meta']['n_views']} sketches, each "
      f"{cfg.sketch_size}x{cfg.sketch_size}")

## The build

One sketch shard per design shard, written with the same marker gating everything else in this project uses: a shard only counts once its `.done.json` exists, so an interrupted build leaves no half-finished shard behind and re-running picks up where it stopped.

Cost, measured on real designs: about 12 KiB per drawing and 35 ms to make one on a CPU core. For the full dataset that is roughly a gigabyte and a half, and an hour or so of CPU - considerably less on a GPU, where a design's whole camera rig goes through in one pass.

In [ ]:
# The build. One sketch shard per design shard, same index, so this is
# resumable at shard granularity: a shard whose done-marker exists is
# skipped, and an interrupted one is rebuilt from scratch. Safe to re-run,
# safe to kill. Nothing but one design is ever held in memory.
build_stats = {}
for split in SPLITS:
    source = shard_dir(cfg, split)
    target = sketch_shard_dir(cfg, split)
    n_shards = len(shards.list_done_shards(source))
    print(f"{split}: {n_shards} design shards -> {target}")
    build_stats[split] = sketchdata.build_sketch_shards(
        source, target, cfg, device=device, progress=tqdm
    )
    print(f"  {build_stats[split]}")

built = sum(s["sketches"] for s in build_stats.values())
print(f"\n{built:,} sketches written this run "
      f"({sum(s['shards_skipped'] for s in build_stats.values())} shards already done)")

In [ ]:
# Open what was just written. Both shard sets extract into ONE local cache,
# so each design ends up with its renders, masks, cameras and now its
# sketches side by side - the sketches ride alongside the 806 MB of
# renders rather than duplicating them.
datasets = {}
for split in SPLITS:
    local = Path(cfg.local_root) / split
    shards.prepare_local(shard_dir(cfg, split), local, progress=tqdm)
    shards.prepare_local(
        sketch_shard_dir(cfg, split), local, progress=tqdm,
        prefix=sketchdata.SKETCH_PREFIX,
    )
    sketchdata.check_params(sketch_shard_dir(cfg, split), cfg)
    datasets[split] = sketchdata.SketchDataset(
        local / "cache", supervision_views=cfg.supervision_views, seed=SEED
    )
    print(f"{split}: {datasets[split].describe()}")

train_ds = datasets[SPLITS[0]]
on_disk = sum(
    p.stat().st_size
    for split in SPLITS
    for p in sketch_shard_dir(cfg, split).glob("*.tar")
)
print(f"\nsketch shards on Drive: {on_disk / 2**30:.2f} GiB")
print(f"total training samples: {sum(len(d) for d in datasets.values()):,}")

In [ ]:
# What a training sample looks like coming out of the Dataset: the input
# drawing on the left, then the ground-truth views it will be scored
# against. Different rows are different (design, input view) pairs - note
# that the same design appears under several inputs, which is the point.
rng = np.random.default_rng(SEED)
picks = rng.choice(len(train_ds), size=3, replace=False)

fig, axes = plt.subplots(
    len(picks), 1 + cfg.supervision_views,
    figsize=(2.0 * (1 + cfg.supervision_views), 2.2 * len(picks)),
)
axes = np.atleast_2d(axes)
for row, index in enumerate(picks):
    sample = train_ds[int(index)]
    axes[row, 0].imshow(sample["input"][0], cmap="gray", vmin=0, vmax=1)
    axes[row, 0].set_ylabel(
        f"{sample['design_id'][:10]}\nview {sample['input_view']}", fontsize=7
    )
    for col in range(cfg.supervision_views):
        view = sample["views"][col].numpy().astype(np.float32) / 255.0
        view = np.where(sample["masks"][col].numpy()[..., None], view, 0.5)
        axes[row, 1 + col].imshow(view)
for ax in axes.ravel():
    ax.set_xticks([])
    ax.set_yticks([])
axes[0, 0].set_title("input sketch", fontsize=9)
for col in range(cfg.supervision_views):
    axes[0, 1 + col].set_title(f"supervision {col}", fontsize=9)
fig.tight_layout()
plt.show()

## Getting it to Ibex

The sketch shards are a sibling of the design shards on Drive, so they move the same way everything else did - rsync from the Mac's synced Drive folder, or rclone from Ibex:

```
rsync -av --progress \
  "~/Library/CloudStorage/GoogleDrive-<you>/My Drive/StarX/sketch_shards/" \
  ibex:/ibex/user/$USER/StarX/sketch_shards/
```

On Ibex the training script finds them next to the design shards under the same data root, extracts both sets into one cache, and checks the stored build parameters against the config before it starts - so a dataset built with different edge settings than the run expects fails loudly rather than training on the wrong drawings.

```
git -C $HOME/StarX pull
torchrun --standalone --nproc_per_node=2 scripts/train_sketch.py \
  --data-root $HOME/StarX/data/StarX --run-name sketch_paper_2gpu
```

Notebook 11 drives the same script's training core interactively, and `scripts/train_sketch.py --help` lists every knob with the paper value as its default.